# Exercise 5: sentiment classification with PyTorch

We want to build a binary sentiment classifier for movie-review snippets using PyTorch tensors and automatic differentiation. Labels are **0 = negative** and **1 = positive**.

### From text to predictions

How could a model distinguish “a wonderful film” from “a terrible film” using numbers? We assign each known word a coordinate and learn how much it supports positive or negative sentiment.

The **training set** determines the vocabulary and model weights. The **validation set** selects an epoch. The **test set** is reserved for the final evaluation.

| Tensor | Shape | Meaning |
|---|---|---|
| `X` | `(B, V)` float32 | B snippets, V vocabulary coordinates |
| `y` | `(B,)` float32 | One binary label per snippet |
| `w`, `b` | `(V,)`, scalar | Trainable weights and bias |
| `z = X @ w + b` | `(B,)` | One score (logit) per snippet |

**Check your understanding:** For a batch of 4 examples and a vocabulary of 100 words, what are the shapes of `X`, `w`, and `z`?

### Read then Run: setup

Run the supplied setup.

The first run downloads the data, which is cached in `data/rt-polaritydata.tar.gz`.

We use Pang and Lee’s **sentence polarity dataset v1.0**, containing 5,331 positive and 5,331 negative movie-review snippets. [Dataset and README](https://www.cs.cornell.edu/people/pabo/movie-review-data/).

The setup tokenizes lowercase words, retaining words such as **not**. It removes exact normalized duplicates, discards conflicting duplicates, creates a seeded stratified 70/15/15 split, and builds a vocabulary of up to 3,000 words occurring at least twice in training text. Unknown words are ignored. The tokenizer drops punctuation and numbers.

In [ ]:
import copy
import hashlib
import io
import math
import random
import re
import tarfile
import urllib.request
from collections import Counter
from pathlib import Path

import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
DEVICE = torch.device("cpu")

def close(actual, expected, atol=1e-6, rtol=1e-5):
    if not isinstance(actual, torch.Tensor):
        actual = torch.as_tensor(actual, dtype=torch.float64)
    expected = torch.as_tensor(expected, dtype=actual.dtype, device=actual.device)
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)

def must_raise(error_type, callback):
    try:
        callback()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("PyTorch:", torch.__version__, "| device:", DEVICE)


DATA_URL = "https://www.cs.cornell.edu/people/pabo/movie-review-data/rt-polaritydata.tar.gz"
DATA_PATH = Path("data/rt-polaritydata.tar.gz")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "PyTorchTeachingNotebook/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            payload = response.read()
        # Validate before caching. Read named members; never extract arbitrary paths.
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            assert any(m.name.endswith("rt-polarity.pos") for m in archive.getmembers())
        DATA_PATH.write_bytes(payload)
    except Exception as exc:
        raise RuntimeError(f"Download {DATA_URL} manually to {DATA_PATH.resolve()}") from exc

raw_records = []
with tarfile.open(DATA_PATH, "r:gz") as archive:
    for label, suffix in [(0, "rt-polarity.neg"), (1, "rt-polarity.pos")]:
        members = [m for m in archive.getmembers() if m.name.endswith(suffix) and m.isfile()]
        assert len(members) == 1, f"Expected exactly one {suffix}"
        with archive.extractfile(members[0]) as stream:
            lines = stream.read().decode("latin-1").strip("\n").split("\n")
        assert len(lines) == 5331, "Unexpected dataset version or incomplete archive"
        raw_records.extend({"text": text.strip(), "label": label} for text in lines if text.strip())

# A supplied data-hygiene step, independent of vocabulary learning.
groups = {}
for row in raw_records:
    key = " ".join(row["text"].lower().split())
    groups.setdefault(key, []).append(row)
records = [rows[0] for rows in groups.values() if len({r["label"] for r in rows}) == 1]
print("Archive SHA-256:", hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())
print("Raw examples:", len(raw_records), "| after deduplication:", len(records))
print("Class counts:", dict(Counter(row["label"] for row in records)))

# Preview handmade examples instead of consulting future test examples
print({"text": "a wonderfully acted film", "label": 1})


def split_indices(rows, seed=SEED):
    rng = random.Random(seed)
    partitions = [[], [], []]
    for label in (0, 1):
        indices = [i for i, row in enumerate(rows) if row["label"] == label]
        rng.shuffle(indices)
        a, b = int(0.70 * len(indices)), int(0.85 * len(indices))
        for target, part in zip(partitions, (indices[:a], indices[a:b], indices[b:])):
            target.extend(part)
    for part in partitions:
        rng.shuffle(part)
    return tuple(partitions)

def tokenize(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())

def build_vocab(texts, min_freq=2, max_size=3000):
    counts = Counter(token for text in texts for token in tokenize(text))
    words = sorted((word for word in counts if counts[word] >= min_freq),
                   key=lambda word: (-counts[word], word))[:max_size]
    return {word: i for i, word in enumerate(words)}

train_idx, val_idx, test_idx = split_indices(records)
train_rows = [records[i] for i in train_idx]
val_rows = [records[i] for i in val_idx]
vocab = build_vocab([row["text"] for row in train_rows])
print("Train / validation / sealed test:", len(train_idx), len(val_idx), len(test_idx))
print("Vocabulary size:", len(vocab))


## Task 1: Represent text with torch

**Why do we need this?**

Raw strings cannot be multiplied by weights. A fixed-length feature vector gives the model numerical evidence for each word.

A **binary bag of words** sets coordinate j to 1 when vocabulary word j appears, and 0 otherwise. Repetitions stay 1. Unknown words are ignored. Rows are snippets and columns are words. Use `float32` because the next steps involve floating-point parameters.

**Your task**

Implement `vectorize(texts, vocab)`: allocate a zero matrix with `torch.zeros`, use the supplied `tokenize`, set known word coordinates to 1. The output shape must be `(len(texts), len(vocab))`. The supplied cell after the checks builds training and validation tensors.

<details><summary>Hint: open after trying</summary>

Loop over `enumerate(texts)` and `set(tokenize(text))`. For known tokens, assign `X[i, vocab[token]] = 1.0`.

</details>

**Explain after checking:** Which information did we lose when a sentence became a bag of words?


In [ ]:
def vectorize(texts, vocab):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
v = {"good": 0, "bad": 1, "not": 2}
close(vectorize(["good good", "not bad", "unknown", ""], v),
      [[1, 0, 0], [0, 1, 1], [0, 0, 0], [0, 0, 0]])
assert vectorize(["good"], v).dtype == torch.float32
assert vectorize([], v).shape == (0, 3)
close(vectorize(["not good"], v), vectorize(["good not"], v))

print("Task 1: all checks passed.")


### Read then Run: assemble the numerical data

The feature and label tensors must use the same row order. Labels have shape `(N,)`, not `(N, 1)`. That distinction avoids accidental broadcasting later. Only parameters will require gradients; inputs and fixed labels do not.

Our reference classifier always predicts the majority **training** label. This baseline reveals how much accuracy is available without reading any words.


In [ ]:
def encode_rows(rows, vocab):
    X = vectorize([row["text"] for row in rows], vocab)
    y = torch.tensor([row["label"] for row in rows], dtype=torch.float32)
    return X, y

X_train, y_train = encode_rows(train_rows, vocab)
X_val, y_val = encode_rows(val_rows, vocab)
majority_label = int(y_train.mean().item() > 0.5)
baseline_val = (y_val == majority_label).float().mean().item()
assert X_train.shape == (len(train_rows), len(vocab))
assert y_train.shape == (len(train_rows),)
print("X_train:", tuple(X_train.shape), "| y_train:", tuple(y_train.shape))
print(f"Majority validation accuracy: {baseline_val:.3f}")


## Task 2: Build a linear sentiment model

**Why do we need this?**

We want positive evidence from some words and negative evidence from others. A weighted sum makes that idea explicit and gives us parameters to learn.

For a batch, `z = X @ w + b`. The real-valued output is a **logit**, `sigmoid(z)` maps it to a positive-class probability. `torch.nn.Parameter` registers tensors as learnable model parameters. Zero initialization is suitable for this linear model.

**Your task**

Complete `BoWClassifier`: create a zero weight vector of length V and a zero scalar bias as `torch.nn.Parameter` objects, then return the batch logits from `forward`. Keep shape `(B,)`, including when B is 1.

<details><summary>Hint: open after trying</summary>

Use `torch.zeros(vocab_size)` for weights, `torch.zeros(())` for the scalar bias, and matrix-vector multiplication with `@`.

</details>

**Explain after checking:** Before training, all logits are 0. What probability does that imply? What would a positive weight on “wonderful” do?


In [ ]:
class BoWClassifier(torch.nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###

    def forward(self, X):
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###


In [ ]:
probe = BoWClassifier(2)
assert len(list(probe.parameters())) == 2
assert probe.w.requires_grad and probe.w.is_leaf
close(probe(torch.ones(3, 2)), [0., 0., 0.])
with torch.no_grad():
    probe.w.copy_(torch.tensor([2., -3.]))
    probe.b.fill_(0.5)
close(probe(torch.tensor([[1., 0.], [0., 1.], [1., 1.]])), [2.5, -2.5, -0.5])
assert probe(torch.ones(1, 2)).shape == (1,)

print("Task 2: all checks passed.")


## Task 3: Give learning a differentiable objective

**Why do we need this?**

A hard correct/incorrect decision does not tell a weight how to change. Binary cross-entropy supplies a smooth signal and penalizes confident mistakes strongly.

For probability $p$ and binary target $y$, $\ell=-y\log(p)-(1-y)\log(1-p)$. In code, use **binary cross-entropy with logits**: PyTorch combines sigmoid and the logarithms stably. This prevents overflow or `log(0)` for extreme scores. Do not apply sigmoid before this loss.

**Your task**

Implement `binary_loss(logits, targets)` with `F.binary_cross_entropy_with_logits`, averaging over examples. The shape check is supplied. Before running the check, predict which loss is larger for a positive example: logit +2 or logit −2.

<details><summary>Hint: open after trying</summary>

Pass the raw logits and float targets to `F.binary_cross_entropy_with_logits(..., reduction="mean")`.

</details>

**Explain after checking:** Why is sigmoid useful when presenting predictions, but unnecessary before this fused loss?


In [ ]:
def binary_loss(logits, targets):
    if logits.ndim != 1 or logits.shape != targets.shape or logits.numel() == 0:
        raise ValueError("Expected matching nonempty vectors")
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
close(binary_loss(torch.zeros(2), torch.tensor([0., 1.])), math.log(2))
assert binary_loss(torch.tensor([2.]), torch.tensor([1.])) < binary_loss(torch.tensor([-2.]), torch.tensor([1.]))
assert torch.isfinite(binary_loss(torch.tensor([-1000., 1000.]), torch.tensor([1., 0.])))
z = torch.tensor([-2., 0., 2.], requires_grad=True)
y = torch.tensor([0., 1., 1.])
binary_loss(z, y).backward()
close(z.grad, (z.detach().sigmoid() - y) / len(y))
print("Logit gradients:", z.grad.tolist())

print("Task 3: all checks passed.")


## Task 4: Make one gradient update

**Why do we need this?**

A loss value alone cannot change the model. Automatic differentiation computes gradients, gradient descent then changes the parameters in the direction that reduces the loss.

One update has four stages: **clear gradients → forward and loss → backward → subtract learning rate × gradient**. PyTorch accumulates gradients across backward calls, so old values must be cleared. Update parameters inside `torch.no_grad()` so the update itself is not added to the computation graph.

**Your task**

Implement `gradient_step(model, X, y, lr)`. Call `model.zero_grad(set_to_none=True)`, compute your loss, call `.backward()`, then update every parameter in place. Return the loss before the update as a Python float. We write this ourselves to see what an optimizer normally does.

<details><summary>Hint: open after trying</summary>

Within `with torch.no_grad():`, loop over `model.parameters()` and use `parameter.add_(parameter.grad, alpha=-lr)`.

</details>

**Explain after checking:** What goes wrong if we forget to clear gradients? What does backward compute, and which line actually learns?


In [ ]:
def gradient_step(model, X, y, lr):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
probe = BoWClassifier(2)
x = torch.tensor([[1., 0.], [0., 1.]])
y = torch.tensor([1., 0.])
initial = gradient_step(probe, x, y, lr=0.2)
close(initial, math.log(2))
close(probe.w, [0.05, -0.05])
close(probe.b, 0.)
assert binary_loss(probe(x), y).item() < initial
# The second update catches stale-gradient accumulation.
expected_second_w = probe.w.detach().clone() - 0.2 * (probe(x).detach().sigmoid() - y) / 2
assert isinstance(gradient_step(probe, x, y, lr=0.2), float)
close(probe.w, expected_second_w)
# A tiny separable problem should be learnable before we try the real corpus.
for _ in range(98):
    gradient_step(probe, x, y, lr=0.5)
with torch.no_grad():
    assert torch.equal((probe(x) >= 0).float(), y)
    assert binary_loss(probe(x), y).item() < 0.1

print("Task 4: all checks passed.")


### Read then Run: batching and evaluation

One update on all examples would work, but **mini-batches** let us update after a small group. An **epoch** visits every training example once. The supplied iterator shuffles examples and preserves the final short batch.

Evaluation must measure a fixed model: `model.eval()` chooses evaluation behavior and `torch.no_grad()` disables gradient tracking. They have different roles, even though this simple model has no dropout. Predict positive at logit ≥ 0, equivalent to probability ≥ 0.5. We weight each batch loss by its number of examples before averaging.

The next task uses these helpers with your gradient step.


In [ ]:
def batches(X, y, batch_size=64, generator=None, shuffle=True):
    order = torch.randperm(len(y), generator=generator) if shuffle else torch.arange(len(y))
    for start in range(0, len(y), batch_size):
        indices = order[start:start + batch_size]
        yield X[indices], y[indices]

def evaluate(model, X, y):
    model.eval()
    total, correct = 0.0, 0
    with torch.no_grad():
        for xb, yb in batches(X, y, batch_size=256, shuffle=False):
            z = model(xb)
            total += binary_loss(z, yb).item() * len(yb)
            correct += ((z >= 0).float() == yb).sum().item()
    return {"loss": total / len(y), "accuracy": correct / len(y)}


## Task 5: Turn one update into an epoch

**Why do we need this?**

One small update rarely learns enough from text. Repeating your gradient step over mini-batches is the essential training loop, validation tells us whether the learned pattern transfers beyond the training examples.

Use `model.train()` and call `gradient_step` for each supplied batch. Return the example-weighted average of the pre-update batch losses. The supplied driver will repeat this for 20 epochs, record end-of-epoch metrics, and restore the checkpoint with the lowest validation loss.

**Your task**

Implement `train_epoch(model, X, y, lr, generator)`. Set training mode, iterate over `batches`, call your gradient step, accumulate `batch_loss * len(yb)`, and divide by the total number of examples. Then run the supplied driver and discuss its curves.

<details><summary>Hint: open after trying</summary>

The generator is created once by the driver; pass it to `batches(X, y, generator=generator)` instead of resetting the seed inside the epoch.

</details>

**Explain after checking:** If training loss improves while validation loss gets worse, would more epochs necessarily help?


In [ ]:
def train_epoch(model, X, y, lr, generator):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
probe = BoWClassifier(2)
x = torch.tensor([[1., 0.], [0., 1.]])
y = torch.tensor([1., 0.])
generator = torch.Generator().manual_seed(SEED)
before = probe.w.detach().clone()
close(train_epoch(probe, x, y, 0.2, generator), math.log(2))
assert not torch.equal(probe.w, before)
assert binary_loss(probe(x), y).item() < math.log(2)
# With learning rate zero, batch weighting must match full-dataset mean loss.
x_more = torch.cat([x.repeat(32, 1), torch.tensor([[5., 0.]])])
y_more = torch.cat([y.repeat(32), torch.tensor([0.])])
close(train_epoch(probe, x_more, y_more, 0.0, generator), binary_loss(probe(x_more), y_more).item())

print("Task 5: all checks passed.")


### Read then Run: train and select a checkpoint

Train with learning rate 0.2, batch size 64, and 20 epochs. Select the epoch with the lowest validation loss. A deep copy preserves the best weights while training continues. The test set remains closed.

Compare the training and validation curves. Do both improve? What does the gap between them suggest?

In [ ]:
model = BoWClassifier(len(vocab))
generator = torch.Generator().manual_seed(SEED)
history, best_state, best_loss, best_epoch = [], None, float("inf"), None
for epoch in range(1, 21):
    train_epoch(model, X_train, y_train, lr=0.2, generator=generator)
    training = evaluate(model, X_train, y_train)
    validation = evaluate(model, X_val, y_val)
    history.append({"epoch": epoch, "train_loss": training["loss"],
                    "val_loss": validation["loss"], "val_accuracy": validation["accuracy"]})
    if validation["loss"] < best_loss:
        best_loss, best_epoch = validation["loss"], epoch
        best_state = copy.deepcopy(model.state_dict())
    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:2d}: train loss={training['loss']:.3f}, "
              f"validation loss={validation['loss']:.3f}, accuracy={validation['accuracy']:.3f}")
model.load_state_dict(best_state)
close(evaluate(model, X_val, y_val)["loss"], best_loss)
assert history[-1]["train_loss"] < history[0]["train_loss"]
print("Selected epoch:", best_epoch, "| majority validation accuracy:", round(baseline_val, 3))

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Read the printed losses above; plotting is optional.")
else:
    fig, ax = plt.subplots(figsize=(6.5, 3))
    ax.plot([r["epoch"] for r in history], [r["train_loss"] for r in history], label="Training")
    ax.plot([r["epoch"] for r in history], [r["val_loss"] for r in history], label="Validation")
    ax.axvline(best_epoch, color="gray", linestyle="--", label="Selected epoch")
    ax.set(xlabel="Epoch", ylabel="Mean cross-entropy", title="Does learning transfer to validation text?")
    ax.legend()
    fig.tight_layout()
    plt.show()


## Task 6: Predict new text and expose a limitation

**Why do we need this?**

The model is only useful if we can apply the same text representation to new inputs. Looking at carefully chosen examples also reveals what this representation cannot express.

For inference, reuse the training vocabulary, compute logits without gradients, apply sigmoid for display, and threshold at 0.5. An unknown-only input has a zero feature vector and therefore receives the bias-only prediction. Probabilities are model estimates, not guarantees of correctness.

**Your task**

Implement `predict_texts(model, texts, vocab)`, returning positive-class probabilities and integer predictions. Then run the supplied examples. Before running the last two examples, discuss whether reversing words can change their bag-of-words vectors.

<details><summary>Hint: open after trying</summary>

Use `model.eval()`; inside `torch.no_grad()`, call `vectorize`, the model, then `torch.sigmoid`. Return probabilities and `(probabilities >= 0.5).long()`.

</details>

**Explain after checking:** Can a larger hidden layer recover word order if two texts already have identical feature vectors?


In [ ]:
def predict_texts(model, texts, vocab):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
probe = BoWClassifier(2)
with torch.no_grad():
    probe.w.copy_(torch.tensor([2., -2.]))
    probe.b.fill_(0.2)
p, pred = predict_texts(probe, ["good", "bad", "unknown"], {"good": 0, "bad": 1})
close(p, torch.sigmoid(torch.tensor([2.2, -1.8, 0.2])))
assert pred.tolist() == [1, 0, 1] and not p.requires_grad
assert p.shape == pred.shape == (3,)

print("Task 6: all checks passed.")


### Read then Run: inspect predictions

Try the supplied examples, then add one of your own. The final two examples contain exactly the same words in different orders, so this model must return identical probabilities.

A bag of words distinguishes “not good” from “good”, but additive weights cannot make “not” reverse the meaning of every word it precedes.

In [ ]:
examples = ["a wonderful and moving film", "a dull and disappointing movie",
            "not a good film", "not a bad film", "qzxv qzxv",
            "the good film is not bad", "the bad film is not good"]
probabilities, predictions = predict_texts(model, examples, vocab)
for text, probability, prediction in zip(examples, probabilities.tolist(), predictions.tolist()):
    print(f"p(positive)={probability:.3f}, label={prediction}: {text}")
close(probabilities[-1], probabilities[-2])


### Final evaluation

Evaluate the selected model on the test set using the fixed vocabulary and 0.5 threshold. Accuracy is the fraction of correct labels. The confusion matrix has gold labels in rows and predictions in columns: `[[TN, FP], [FN, TP]]`.

Use training and validation data for model selection, not test results. This random snippet split may contain related snippets from the same movie and is not a standard benchmark split.

In [ ]:
test_rows = [records[i] for i in test_idx]
X_test, y_test = encode_rows(test_rows, vocab)
final_metrics = evaluate(model, X_test, y_test)
model.eval()
with torch.no_grad():
    pred = model(X_test) >= 0
    gold = y_test == 1
    tn, fp = int((~pred & ~gold).sum()), int((pred & ~gold).sum())
    fn, tp = int((~pred & gold).sum()), int((pred & gold).sum())
confusion_matrix = [[tn, fp], [fn, tp]]
assert sum(map(sum, confusion_matrix)) == len(y_test)
print("Final test accuracy:", round(final_metrics["accuracy"], 3))
print("Majority test accuracy:", round((y_test == majority_label).float().mean().item(), 3))
print("Confusion matrix [[TN, FP], [FN, TP]]:", confusion_matrix)


### References

- Pang and Lee (2005), *Seeing stars: Exploiting class relationships for sentiment categorization with respect to rating scales*. [Sentence polarity dataset v1.0](https://www.cs.cornell.edu/people/pabo/movie-review-data/).
- PyTorch: [autograd](https://docs.pytorch.org/docs/stable/autograd.html) and [binary cross-entropy with logits](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.binary_cross_entropy_with_logits.html).